In [ ]:
!pip install librosa soundfile scikit-learn numpy pandas joblib openai-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.3 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=7f267a2fa2b417926d76d98b4da0c65e781350ddae2cec7978a9d17f7faeaa06
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import os
import numpy as np
import librosa
import pandas as pd
import joblib
import base64
import io
import whisper

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [ ]:
def extract_features(audio, sr):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

    spectral_centroid = np.mean(
        librosa.feature.spectral_centroid(y=audio, sr=sr)
    )
    spectral_bandwidth = np.mean(
        librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    )
    zcr = np.mean(librosa.feature.zero_crossing_rate(audio))

    pitches, _ = librosa.piptrack(y=audio, sr=sr)
    pitch_vals = pitches[pitches > 0]

    pitch_mean = np.mean(pitch_vals) if len(pitch_vals) > 0 else 0
    pitch_std = np.std(pitch_vals) if len(pitch_vals) > 0 else 0

    return [
        np.mean(mfcc),
        np.std(mfcc),
        spectral_centroid,
        spectral_bandwidth,
        zcr,
        pitch_mean,
        pitch_std
    ]


In [ ]:
X = []
y = []

base_path = "test2.wav"                #give a path where you have the set of wav files

for label, folder in enumerate(["human", "ai"]):
    folder_path = os.path.join(base_path, folder)

    for file in os.listdir(folder_path):
        if file.endswith(".wav"):
            file_path = os.path.join(folder_path, file)

            audio, sr = librosa.load(file_path, sr=16000)
            features = extract_features(audio, sr)

            X.append(features)
            y.append(label)

X = np.array(X)
y = np.array(y)

print("Total samples:", len(X))


NotADirectoryError: [Errno 20] Not a directory: 'test2.wav/human'

In [2]:
# Run this only when you get errors
"""from google.colab import drive
drive.mount('/content/drive')
"""

"from google.colab import drive\ndrive.mount('/content/drive')\n"

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Human", "AI"]))


In [ ]:
joblib.dump(model, "model.pkl")
print("Model saved as model.pkl")


In [ ]:
whisper_model = whisper.load_model("small")


In [ ]:
def detect_language(audio):
    audio = whisper.pad_or_trim(audio)
    mel = whisper.log_mel_spectrogram(audio).to(whisper_model.device)
    _, probs = whisper_model.detect_language(mel)

    language = max(probs, key=probs.get)
    confidence = probs[language]

    return language, round(float(confidence), 3)
